# YOLO-World로 도메인 갭 넘기 — 오픈보캡 검출 실습

[vision_counting_pipeline](vision_counting_pipeline_practice.ipynb)의 도메인 갭 셀에서 확인한 문제를 그대로 이어받습니다:

```
YOLO11n(COCO 고정 80클래스)을 시설재배 이미지에 적용
→ 검출 7개, 전부 오분류(해당 클래스가 COCO에 아예 없음)
→ "클래스 정의부터 다시 해야 하는 문제"
```

이 노트북은 **같은 이미지**에 **오픈보캡(open-vocabulary) 검출기**를 적용해 그 문제를 넘어갑니다.
YOLO-World는 텍스트 프롬프트로 클래스를 즉석에서 지정합니다 — 재학습이 아니라
**제로샷(zero-shot) 프롬프팅**입니다. 이 모델군의 "개발 과정"은 학습 루프가 아니라
**어떤 프롬프트가 잘 잡는지 비교·설계하는 것**입니다.

> `yolov8s-world.pt`는 field2scene(개인 프로젝트, `"tomato"` 프롬프트로 도메인 갭 측정)에서
> 이미 실사용한 모델입니다. 이 노트북은 그 방식을 Jetson 위에서 직접 재현합니다.

## 0단계. 환경 준비

`ultralytics`는 이미 설치돼 있습니다(vision_counting_pipeline 노트북에서 `--no-deps`로 설치).
YOLO-World 가중치만 추가로 받으면 됩니다 — 최초 1회 자동 다운로드(`yolov8s-world.pt`, 약 25MB).

In [ ]:
# [코드 1] 환경 확인 + YOLO-World 로드
import os
from importlib.metadata import version
import torch
from ultralytics import YOLO

BEFORE = version("torch")
print("torch:", BEFORE, "| CUDA:", torch.cuda.is_available())

WORK = os.path.expanduser("~/yolo_world_lab")
os.makedirs(WORK, exist_ok=True)

model = YOLO("yolov8s-world.pt")            # 최초 1회 자동 다운로드
print("모델 로드 완료:", type(model.model).__name__)

AFTER = version("torch")
assert AFTER == BEFORE, f"torch가 변경됨! {BEFORE} -> {AFTER}"
print("torch 무사, 작업 디렉토리:", WORK)

## 1단계. 대조군

먼저 YOLO-World가 정상적으로 동작하는지, 이미 정답을 아는 이미지로 확인합니다.
`vision_counting_pipeline`의 [코드 2]와 같은 이미지입니다 — 그때는 COCO 고정 클래스(`apple`,
`orange`, `banana`)로 10개를 찾았습니다. 여기서는 **같은 이름을 텍스트 프롬프트로 지정**해서
동등한 조건을 만듭니다.

In [ ]:
# [코드 2] 1단계 — 대조군: 가판대 이미지에 프롬프트 지정
import os, urllib.request, collections
import cv2, matplotlib.pyplot as plt

def show(img_bgr, title="", figsize=(9, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

CANDIDATES = [
    "http://images.cocodataset.org/val2017/000000560256.jpg",
    "http://images.cocodataset.org/val2017/000000303566.jpg",
]
stall_path = os.path.join(WORK, "sample_fruit.jpg")
if not os.path.exists(stall_path):
    for url in CANDIDATES:
        try:
            urllib.request.urlretrieve(url, stall_path)
            print("샘플 이미지:", url)
            break
        except Exception as e:
            print("실패:", url, e)

model.set_classes(["apple", "orange", "banana"])       # 텍스트 프롬프트로 클래스 지정
r = model(stall_path, verbose=False)[0]
counts = collections.Counter(r.names[int(c)] for c in r.boxes.cls)
n_stall = len(r.boxes)

print("검출 결과:", dict(counts))
print(f"[YOLO-World] 검출 수: {n_stall}  (YOLO11n/COCO는 10개였음)")
show(r.plot(), f"YOLO-World, prompt=[apple,orange,banana]: {n_stall} boxes")

## 2단계. 도메인 갭 이미지

[vision_counting_pipeline](vision_counting_pipeline_practice.ipynb)의 도메인 갭 셀에서 썼던
**같은 시설재배 이미지**입니다. 그때 COCO 사전학습 YOLO11n은 검출 7개, **전부 오분류**였습니다
(사과·오렌지로 착각) — 해당 클래스가 COCO 80개 중에 아예 없었기 때문입니다.

이번엔 `model.set_classes(["tomato"])` 한 줄로 그 클래스를 즉석에서 만듭니다.
재학습도, 라벨링도 없습니다.

In [ ]:
# [코드 3] 2단계 — 도메인 갭 이미지에 "tomato" 프롬프트
import urllib.request

FIELD_URLS = [
    "https://raw.githubusercontent.com/laboroai/LaboroTomato/master/examples/raw_IMG_1066.png",
    "https://raw.githubusercontent.com/laboroai/LaboroTomato/master/examples/raw_IMG_1246.png",
]
field_path = os.path.join(WORK, "field.png")
if not os.path.exists(field_path):
    for url in FIELD_URLS:
        try:
            urllib.request.urlretrieve(url, field_path)
            print("현장 이미지:", url.rsplit("/", 1)[-1])
            break
        except Exception as e:
            print("실패:", url, e)

model.set_classes(["tomato"])
rf = model(field_path, verbose=False)[0]
n_field = len(rf.boxes)

# YOLO11n(COCO)의 실측 — vision_counting_pipeline [코드 3]에서 이미 측정한 값
YOLO11N_COCO_DETECTED = 7      # 전부 오분류(apple/orange)였음

print(f"[YOLO-World, prompt='tomato'] 검출 수: {n_field}")
print(f"[YOLO11n, COCO 80클래스]        검출 수: {YOLO11N_COCO_DETECTED}  (전부 오분류였음)")
if n_field:
    print(f"\n프롬프트만으로 재학습 없이 {n_field}개를 '토마토'로 정확히 찾았습니다.")
    print("클래스 정의 문제였다는 걸 거꾸로 증명하는 결과입니다 —")
    print("모델을 안 바꾸고 무엇을 찾을지만 알려줬는데 결과가 바뀌었습니다.")
else:
    print("\n0개입니다. 프롬프트 표현을 바꿔야 할 수 있습니다 — 다음 셀에서 시도합니다.")

show(cv2.imread(field_path) if n_field == 0 else rf.plot(),
     f"YOLO-World, prompt='tomato': {n_field} boxes", figsize=(6, 8))

## 3단계. 프롬프트 설계

파인튜닝이 없으니 성능을 좌우하는 건 **프롬프트 표현**입니다. 같은 대상이라도
단어 선택에 따라 검출 개수·신뢰도가 달라집니다. 몇 가지를 비교합니다:

- `"tomato"` — 가장 단순한 클래스명
- `"red tomato"` — 색을 명시 (완숙만 잡힐 가능성)
- `"tomato, green tomato"` — 여러 프롬프트를 동시에 등록
- `"ripe tomato on the vine"` — 문맥을 넣은 서술형 프롬프트

실무에서는 이 비교 자체가 "모델 개발"입니다 — 데이터를 모으고 라벨링하는 대신,
**어떤 텍스트가 이 도메인을 가장 잘 짚는지**를 검증합니다.

In [ ]:
# [코드 4] 3단계 — 프롬프트별 검출 결과 비교
PROMPTS = [
    ["tomato"],
    ["red tomato"],
    ["tomato", "green tomato"],
    ["ripe tomato on the vine"],
]

results = []
for classes in PROMPTS:
    model.set_classes(classes)
    r = model(field_path, verbose=False)[0]
    n = len(r.boxes)
    conf = float(r.boxes.conf.mean()) if n else 0.0
    label = " + ".join(classes)
    results.append((label, n, conf))
    print(f"{label:32s} 검출 {n:>2}개   평균 신뢰도 {conf:.2f}")

best = max(results, key=lambda x: x[1])
print(f"\n가장 많이 찾은 프롬프트: '{best[0]}' ({best[1]}개)")

fig, ax = plt.subplots(figsize=(8, 4))
labels = [r[0] for r in results]
counts = [r[1] for r in results]
ax.bar(labels, counts, color="#4c72b0")
ax.set_ylabel("detections"); ax.set_title("YOLO-World: detections by prompt")
plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.show()

## 정리

1. **도메인 갭의 두 번째 해법** — [vision_counting_pipeline](vision_counting_pipeline_practice.ipynb)에서는
   "클래스 정의부터 다시 해야 한다"가 결론이었다. 이 노트북은 그 재정의를 **재학습이 아니라
   텍스트 프롬프트**로 해냈다. 데이터 수집·라벨링·학습 루프 없이 클래스를 즉석에서 추가할 수 있다.
2. **다만 만능은 아니다** — 오픈보캡 모델은 폐쇄형 검출기보다 느리고, 프롬프트 표현에 따라
   결과가 흔들린다. 정확도가 결정적인 배포라면 결국 도메인 데이터로 파인튜닝한 폐쇄형 모델이
   더 안정적이다. 프로토타이핑·희귀 클래스 대응에는 오픈보캡이, 대량 배포에는 폐쇄형+경량화가 맞다.
3. **이 모델군의 "개발"은 프롬프트 설계다** — 학습 루프 대신 어떤 표현이 이 도메인을
   가장 잘 짚는지 비교하는 과정 자체가 개발 사이클이다.

### 직접 해보기

- 검출된 토마토 crop에 [tomato_ripeness_training](tomato_ripeness_training_practice.ipynb)의
  분류기를 이어붙여 완전한 end-to-end 데모 만들기 (검출은 YOLO-World, 판정은 학습한 ResNet18)
- YOLO-World도 TensorRT로 배포해 속도 비교 (오픈보캡의 실시간 가능성 확인)
- 다른 도메인 갭 이미지(적엽 흔적, 생장점 등)에도 같은 프롬프트 전략 적용